# 03 — Calibration: Reliability Diagram & ECE

Compares the calibration of the baseline EEGNet softmax against MC-dropout confidences using Expected Calibration Error (ECE) and reliability diagrams.

**Gate:** ECE reduction > 0% (MC dropout should improve calibration). The saved `calibration_curve.png` is the resume artifact.

Requires `02_training.ipynb` to have been run (produces `mc_confidences.npy`, `mc_preds.npy`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N_PASSES = 50
probs_baseline = np.load("checkpoints/test_probs_baseline.npy")
y_test = np.load("checkpoints/test_labels.npy")
mc_confidences = np.load("checkpoints/mc_confidences.npy")
mc_preds = np.load("checkpoints/mc_preds.npy")

In [ ]:
def reliability_diagram(confidences, correct, n_bins=10, label=""):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        if mask.sum() > 0:
            bin_accs.append(correct[mask].mean())
            bin_confs.append(confidences[mask].mean())
            bin_sizes.append(mask.sum())
    return np.array(bin_confs), np.array(bin_accs), np.array(bin_sizes)

In [ ]:
def ece(confidences, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    total = len(confidences)
    ece_val = 0.0
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean()
            conf = confidences[mask].mean()
            ece_val += (mask.sum() / total) * abs(acc - conf)
    return ece_val

In [ ]:
baseline_conf = probs_baseline.max(axis=1)
baseline_preds = probs_baseline.argmax(axis=1)
baseline_correct = (baseline_preds == y_test)
mc_correct = (mc_preds == y_test)

ece_baseline = ece(baseline_conf, baseline_correct)
ece_mc = ece(mc_confidences, mc_correct)
print(f"ECE baseline: {ece_baseline:.4f}")
print(f"ECE MC dropout: {ece_mc:.4f}")
print(f"ECE reduction: {(ece_baseline - ece_mc) / ece_baseline * 100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, confs, correct, label in [
    (axes[0], baseline_conf, baseline_correct, "Baseline EEGNet"),
    (axes[1], mc_confidences, mc_correct, f"MC Dropout (N={N_PASSES})"),
]:
    bc, ba, _ = reliability_diagram(confs, correct)
    ax.plot([0, 1], [0, 1], 'k--', label="Perfect calibration")
    ax.plot(bc, ba, 'o-', label=label)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{label}\nECE={ece(confs, correct):.4f}")
    ax.legend()

plt.tight_layout()
plt.savefig("checkpoints/calibration_curve.png", dpi=150)
plt.show()
# This image is your resume artifact